# Stage 4 — Post-Response Re-Extraction (for RQ3)

Two changes from Stage 2, both necessary:

**1. A POST-response window.** The Stage 2 epochs span -450 to -150 ms relative to
the response, i.e. entirely *before* it. The error-monitoring signals that RQ3 depends on
sit *after* the response:

| Component | Timing (relative to response) | Meaning |
|---|---|---|
| ERN (error-related negativity) | 0 to +100 ms, fronto-central | automatic error detection |
| Pe (error positivity) | +200 to +400 ms, parietal | conscious error *awareness* |

The overconfidence hypothesis is that on high-confidence errors the **Pe is attenuated**,
because the person never realised they erred. None of that is in the Stage 2 file.
Here we extract **-100 to +600 ms around the response**, which covers a short pre-response
baseline plus both components.

**2. Save `excludedchannels` per subject.** Stage 2 cropped every subject to the first
58 channels. Because different subjects had different electrodes removed, row *i* refers
to different scalp locations in different subjects. This was verified statistically:
subject pairs that both had the full 64-channel montage shared spatial correlation
structure at r=0.226, versus r=0.105 for all other pairs (Mann-Whitney p=8.7e-12).
Recording `excludedchannels` lets us map every subject onto a common 64-channel montage,
which is what makes pooling the 294 overconfidence trials valid.

Run this once; it produces a second `.npz` alongside the Stage 2 one.

In [ ]:
# 1. Mount Drive and install Octave
from google.colab import drive
drive.mount('/content/drive')

!apt-get update -qq
!apt-get install -y octave -qq
print("Octave ready")

In [ ]:
# 2. Configuration -- EDIT DATA_DIR
DATA_DIR = "/content/drive/MyDrive/PATH_TO_YOUR_EEG_FOLDER"   # <-- EDIT
OUT_PATH = "/content/drive/MyDrive/eeg_epochs_postresp.npz"

# Response-locked window: a short pre-response baseline plus ERN and Pe
PRE_MS   = 100    # ms BEFORE the response (baseline)
POST_MS  = 600    # ms AFTER the response (covers ERN ~0-100, Pe ~200-400)
ORIG_FS  = 1000
TARGET_FS = 250

DS = ORIG_FS // TARGET_FS
TOTAL_MS = PRE_MS + POST_MS
N_OUT = int(TOTAL_MS / 1000 * TARGET_FS)
print(f"Window: -{PRE_MS} to +{POST_MS} ms around response = {TOTAL_MS} ms")
print(f"Downsample {ORIG_FS}->{TARGET_FS} Hz, {N_OUT} samples per trial")

In [ ]:
# 3. Pair up data and events files
import os, glob, re

data_files  = sorted(glob.glob(os.path.join(DATA_DIR, "**", "EEG_data_sub-*.mat"), recursive=True))
event_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "EEG_events_sub-*.mat"), recursive=True))
pat = re.compile(r"sub-(\d+)_run-(\d+)")

def key_of(p):
    m = pat.search(os.path.basename(p))
    return (m.group(1), m.group(2)) if m else None

data_map  = {key_of(f): f for f in data_files  if key_of(f)}
event_map = {key_of(f): f for f in event_files if key_of(f)}
paired = sorted(set(data_map) & set(event_map))
print(f"{len(paired)} subject/run pairs found")

In [ ]:
# 4. Octave loader -- now ALSO extracting excludedchannels
import subprocess
import numpy as np
from scipy.io import loadmat

def load_eeg_and_channels(mat_path, tmp="/content/tmp_Y.mat"):
    """Return (Y, excludedchannels). The excluded-channel list is what lets us
    reconstruct which physical electrode each row corresponds to."""
    if os.path.exists(tmp):
        os.remove(tmp)
    # Some files may not carry the field under the same name; guard for it.
    cmd = (
        f"load('{mat_path}', 'EEGdata'); Y=EEGdata.Y; "
        f"if isfield(EEGdata,'excludedchannels'); exch=EEGdata.excludedchannels; "
        f"else; exch=[]; end; "
        f"save('-mat7-binary', '{tmp}', 'Y', 'exch');"
    )
    r = subprocess.run(["octave","--no-gui","--quiet","--eval",cmd],
                       capture_output=True, text=True)
    if not os.path.exists(tmp):
        raise RuntimeError(f"Octave failed for {mat_path}:\n{r.stderr[:600]}")
    m = loadmat(tmp)
    Y = m["Y"]
    exch = np.array(m.get("exch", [])).squeeze()
    exch = np.atleast_1d(exch).astype(int) if exch.size else np.array([], dtype=int)
    os.remove(tmp)
    return Y, exch

In [ ]:
# 5. Inspect what fields actually exist, on ONE file, before the full run.
# If 'excludedchannels' is absent the fallback below is used instead.
probe = data_map[paired[0]]
print("Probing:", probe)
!octave --no-gui --quiet --eval "load('{probe}','EEGdata'); disp(fieldnames(EEGdata));"

In [ ]:
# 6. MAIN LOOP -- response-locked epochs, keeping channel identity
import gc, time

X_all, subj_all, run_all = [], [], []
conf_all, acc_all, choice_all, dotdir_all, rt_all, tidx_all = [], [], [], [], [], []
chan_ids_all = []          # physical electrode index for each row, per trial
excluded_by_run = {}
log = []
t0 = time.time()

for i, key in enumerate(paired):
    subj, run = key
    print(f"[{i+1}/{len(paired)}] sub-{subj} run-{run} ...", end=" ", flush=True)
    try:
        ev = loadmat(event_map[key])
        tstim = ev["tstim"].squeeze().astype(float)
        RT    = ev["RT"].squeeze().astype(float)
        confidence = ev["confidence"].squeeze().astype(int)
        accuracy   = ev["accuracy"].squeeze().astype(int)
        choice     = ev["choice"].squeeze().astype(int)
        dotdir     = ev["dotdirection"].squeeze().astype(int)

        EEG, exch = load_eeg_and_channels(data_map[key])
        n_ch, n_samp = EEG.shape
        excluded_by_run[f"{subj}_{run}"] = exch

        # Physical electrode id for each stored row: all electrodes minus excluded ones
        MONTAGE = 64
        keep = np.array([c for c in range(1, MONTAGE + 1) if c not in set(exch.tolist())])
        if len(keep) != n_ch:
            # Fallback: cannot resolve identity, tag rows as unknown (-1)
            keep = np.full(n_ch, -1)

        n_kept = n_skip = 0
        for t in range(len(tstim)):
            resp = tstim[t] + RT[t]                 # authoritative response time
            ws = int(round(resp - PRE_MS))
            we = ws + TOTAL_MS
            if ws < 0 or we > n_samp:
                n_skip += 1; continue
            seg = EEG[:, ws:we]
            if seg.shape[1] != TOTAL_MS:
                n_skip += 1; continue
            seg = seg[:, :DS * N_OUT].reshape(n_ch, N_OUT, DS).mean(axis=2)

            X_all.append(seg.astype(np.float32))
            chan_ids_all.append(keep)
            subj_all.append(subj); run_all.append(run)
            conf_all.append(confidence[t]); acc_all.append(accuracy[t])
            choice_all.append(choice[t]); dotdir_all.append(dotdir[t])
            rt_all.append(RT[t]); tidx_all.append(t)
            n_kept += 1

        log.append((subj, run, len(tstim), n_kept, n_skip, n_ch, len(exch)))
        print(f"kept {n_kept}/{len(tstim)}, ch={n_ch}, excluded={list(exch)}")
        del EEG; gc.collect()
    except Exception as e:
        print(f"FAILED: {e}")
        log.append((subj, run, -1, 0, -1, -1, -1))

print(f"\nElapsed {(time.time()-t0)/60:.1f} min;  epochs collected: {len(X_all)}")

In [ ]:
# 7. Project every subject onto a COMMON 64-channel montage.
# Rows for electrodes a subject does not have are filled with NaN, so no
# silent misalignment can occur -- missing is explicitly missing.
MONTAGE = 64
n_ep = len(X_all)
Xc = np.full((n_ep, MONTAGE, N_OUT), np.nan, dtype=np.float32)

unresolved = 0
for i, (seg, ids) in enumerate(zip(X_all, chan_ids_all)):
    if np.all(ids == -1):
        unresolved += 1
        continue
    for r, cid in enumerate(ids):
        if cid >= 1:
            Xc[i, cid - 1, :] = seg[r]

print(f"Aligned array: {Xc.shape}")
print(f"Epochs whose channel identity could NOT be resolved: {unresolved}")
frac_nan = np.isnan(Xc).mean(axis=(0, 2))
print("\nPer-electrode missing fraction (should be small and electrode-specific):")
for c in range(MONTAGE):
    if frac_nan[c] > 0:
        print(f"  electrode {c+1}: missing in {100*frac_nan[c]:.1f}% of epochs")

In [ ]:
# 8. Save
np.savez_compressed(
    OUT_PATH,
    X=Xc,
    subject=np.array(subj_all), run=np.array(run_all),
    confidence=np.array(conf_all), accuracy=np.array(acc_all),
    choice=np.array(choice_all), dotdirection=np.array(dotdir_all),
    RT=np.array(rt_all), trial_index=np.array(tidx_all),
    pre_ms=PRE_MS, post_ms=POST_MS, fs=TARGET_FS, montage=MONTAGE,
)
print(f"Saved {OUT_PATH}  ({os.path.getsize(OUT_PATH)/1e6:.1f} MB)")
if os.path.getsize(OUT_PATH) > 480e6:
    print("WARNING: >480 MB. Drop TARGET_FS to 200 or POST_MS to 500 and re-run.")

In [ ]:
# 9. Report -- paste this back
from collections import Counter
subj_arr=np.array(subj_all); acc_arr=np.array(acc_all); conf_arr=np.array(conf_all)
n=len(acc_arr)
print("=== RETENTION ===")
print(f"{'sub':>4} {'run':>4} {'tot':>5} {'kept':>5} {'skip':>5} {'ch':>4} {'nexcl':>6}")
for row in log: print("{:>4} {:>4} {:>5} {:>5} {:>5} {:>4} {:>6}".format(*row))
print(f"\nEpochs: {n};  subjects: {len(set(subj_all))}")

is_low=np.isin(conf_arr,[1,2,3]); is_high=np.isin(conf_arr,[7,8,9])
print(f"\n=== 4-WAY TABLE ===")
print(f"{'':>12} {'Low':>8} {'High':>8}")
for av,al in [(1,'Correct'),(0,'Incorrect')]:
    m=acc_arr==av
    print(f"{al:>12} {int((m&is_low).sum()):>8} {int((m&is_high).sum()):>8}")
oc=(acc_arr==0)&is_high
print(f"\nOverconfidence cell: {int(oc.sum())} trials from {len(set(subj_arr[oc]))} subjects")
print("Per-subject:", dict(sorted(Counter(subj_arr[oc]).items())))

## What to send back

The output of **cell 5** (field names -- tells us whether `excludedchannels` exists
under that name), **cell 7** (alignment report), and **cell 9** (retention + 4-way table).

If cell 5 shows the field is named something else, tell me and I'll adjust the loader
in cell 4 before you run the long loop.